In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# !pip install ipykernel ipywidgets --break-system-packages

In [14]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor

In [15]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [16]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.10.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Name: NVIDIA RTX 6000 Ada Generation
VRAM: 47.5 GB


In [17]:
notebook_login()

In [18]:
MODEL_ID = "google/gemma-4-E2B-it"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound-e2b"
LOCAL_PATH = "./local_model-e2b"

In [19]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Fetching 9 files:   0%|                                   | 0/9 [00:00<?, ?it/s]Still waiting to acquire lock on local_model-e2b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on local_model-e2b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on local_model-e2b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on local_model-e2b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 9 files: 100%|███████████████████████████| 9/9 [00:17<00:00,  1.91s/it]
Download complete: : 10.3GB [00:17, 550MB/s]              /workspace/local_model-e2b
Download complete: : 10.3GB [00:17, 595MB/s]


In [20]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [21]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=Fals

In [ ]:
layer_config = {}
for name, _ in model.named_modules():
    if name.startswith(("model.vision_tower", "model.audio_tower",
                        "model.multi_modal_projector", "model.audio_projector")):
        layer_config[name] = {"bits": 32}

print(f"Skipping {len(layer_config)} non-LM modules")


Successfully locked 895 Gemma-4 multi-modal modules at 16-bit precision.


In [ ]:
TUNING_CONFIG = {
    "group_size": 128,
    "sym": True,
    "iters": 0,               # RTN mode — required for Gemma 4
    "disable_opt_rtn": True,
    "nsamples": 256,
    "seqlen": 2048,
    "low_gpu_mem_usage": False,
    "quant_nontext_module": False,
    "layer_config": layer_config,
}

In [25]:
def push_to_hub(local_dir, repo_name, token):
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")
    try:
        api = HfApi()
        create_repo(full_repo_id, repo_type="model", exist_ok=True, private=False, token=token)
        api.upload_folder(folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token)
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [26]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-06-06 12:18:12 INFO entry.py L587: Using MLLM mode for multimodal model.


In [27]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,auto_awq", inplace=True
)

2026-06-06 12:18:14 WARNING special_model_handler.py L383: Applying a monkey patch to Gemma4 to reduce RAM usage. This patch has only been validated with limited Transformers versions. Proceed with caution.
2026-06-06 12:18:16 WARNING utils.py L508: 'model.vision_tower.encoder.rotary_emb' exists in the model but is not a supported quantization target in the current scheme, ignoring its setting in `layer_config`
2026-06-06 12:18:16 WARNING utils.py L508: 'model.vision_tower.encoder.layers.0.self_attn.q_norm' exists in the model but is not a supported quantization target in the current scheme, ignoring its setting in `layer_config`
2026-06-06 12:18:16 WARNING utils.py L508: 'model.vision_tower.encoder.layers.0.self_attn.k_norm' exists in the model but is not a supported quantization target in the current scheme, ignoring its setting in `layer_config`
2026-06-06 12:18:16 WARNING utils.py L508: 'model.vision_tower.encoder.layers.0.self_attn.v_norm' exists in the model but is not a supporte

Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

packing: 100%|██████████| 524/524 [00:07<00:00, 68.16it/s]


Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

2026-06-06 12:38:12 INFO export.py L166: Saving quantized model to auto_awq format
packing: 100%|██████████| 524/524 [00:06<00:00, 78.29it/s]


Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

2026-06-06 12:38:43 INFO device.py L1840: 'peak_ram': 27.45GB, 'peak_vram': 24.37GB


(Gemma4ForConditionalGeneration(
   (model): Gemma4Model(
     (vision_tower): Gemma4VisionModel(
       (patch_embedder): Gemma4VisionPatchEmbedder(
         (input_proj): Linear(in_features=768, out_features=768, bias=False)
       )
       (encoder): Gemma4VisionEncoder(
         (rotary_emb): Gemma4VisionRotaryEmbedding()
         (layers): ModuleList(
           (0-15): 16 x Gemma4VisionEncoderLayer(
             (self_attn): Gemma4VisionAttention(
               (q_proj): Gemma4ClippableLinear(
                 (linear): Linear(in_features=768, out_features=768, bias=False)
               )
               (k_proj): Gemma4ClippableLinear(
                 (linear): Linear(in_features=768, out_features=768, bias=False)
               )
               (v_proj): Gemma4ClippableLinear(
                 (linear): Linear(in_features=768, out_features=768, bias=False)
               )
               (o_proj): Gemma4ClippableLinear(
                 (linear): Linear(in_features=768, out_f

In [28]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [30]:
if hf_token:
    # Push auto_round format
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-e2b-w4g128/auto-round-auto-gptq"),
        f"{base_name}-W4A16-AutoRound",
        hf_token
    )
    # Push auto_gptq format (vLLM compatible)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-e2b-w4g128/auto-gptq"),
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
    # Push auto_awq format (vLLM compatible)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-e2b-w4g128/auto-awq"),
        f"{base_name}-W4A16-AutoRound-AWQ",
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")



[Hub] Pushing ./AutoRound-e2b/local_model-e2b-w4g128/auto-round-auto-gptq to Vishva007/gemma-4-E2B-it-W4A16-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/gemma-4-E2B-it-W4A16-AutoRound

[Hub] Pushing ./AutoRound-e2b/local_model-e2b-w4g128/auto-gptq to Vishva007/gemma-4-E2B-it-W4A16-AutoRound-GPTQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/gemma-4-E2B-it-W4A16-AutoRound-GPTQ

[Hub] Pushing ./AutoRound-e2b/local_model-e2b-w4g128/auto-awq to Vishva007/gemma-4-E2B-it-W4A16-AutoRound-AWQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/gemma-4-E2B-it-W4A16-AutoRound-AWQ
